In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import root_mean_squared_error

In [ ]:
series = df['Milk']
result = seasonal_decompose(series, model='additive',period=12)
result.plot()
plt.show()

series = df['Milk']
result = seasonal_decompose(series, model='multiplicative',period=12)
result.plot()
plt.show()

## Centered Rolling Mean
y = df['Milk']
fcast = y.rolling(3,center=True).mean()
plt.plot(y, label='Original Data')
plt.plot(fcast, label='Centered Moving Average')
plt.legend(loc='best')
plt.show()

#Train Test Split for TS
y_train = df['Milk'].iloc[:-12]
y_test = df['Milk'].iloc[-12:]

# Trailing Rolling Mean
span=5
fcast = y_train.rolling(span).mean()
MA = fcast.iloc[-1]
MA_series = pd.Series(MA.repeat(len(y_test)))
MA_fcast = pd.concat([fcast,MA_series], ignore_index=True)
rmse = root_mean_squared_error(y_test, MA_series)
plt.plot(y_train, label='Train')
plt.plot(y_test, label='Test')
plt.plot(MA_fcast, label='Rolling Average Forecast')
plt.title(f"RMSE = {rmse:.2f}")
plt.legend(loc='best')
plt.show()

plt.plot(y_test, label='Test')
MA_series.index = y_test.index
plt.plot(MA_series, label='Rolling Average Forecast')
plt.title(f"RMSE = {rmse:.2f}")
plt.legend(loc='best')
plt.show()

In [ ]:
# Simple Exponential Smoothing

from statsmodels.tsa.api import SimpleExpSmoothing

alpha = 0.2
ses = SimpleExpSmoothing(y_train)
fit1 = ses.fit(smoothing_level=alpha)
fcast1 = fit1.forecast(len(y_test))
y_test.plot(color="pink", label='Test')
fcast1.plot(color="purple", label='Forecast')
rmse = root_mean_squared_error(y_test, fcast1)
plt.title(f"RMSE = {rmse:.2f}")
plt.legend(loc='best')
plt.show()

from ipywidgets import interact, widgets

ses = SimpleExpSmoothing(y_train)
def simple_exp(alpha):
    fit1 = ses.fit(smoothing_level=alpha)
    fcast1 = fit1.forecast(len(y_test))
    y_test.plot(color="pink", label='Test')
    fcast1.plot(color="purple", label='Forecast')
    rmse = root_mean_squared_error(y_test, fcast1)
    plt.title(f"RMSE = {rmse:.2f}, alpha = {alpha:.2f}")
    plt.legend(loc='best')
    plt.show()
widgets.interact(simple_exp, alpha=(0.01, 1, 0.01))

In [ ]:
# Holt's Linear Trend

from statsmodels.tsa.api import Holt

alpha = 0.8
beta = 0.02
holt = Holt(y_train)
fit1 = holt.fit(smoothing_level=alpha, smoothing_trend=beta)
fcast1 = fit1.forecast(len(y_test))
y_test.plot(color="pink", label='Test')
fcast1.plot(color="purple", label='Forecast')
rmse = root_mean_squared_error(y_test, fcast1)
plt.title(f"RMSE = {rmse:.2f}")
plt.legend(loc='best')
plt.show()

holt = Holt(y_train)
def holt_linear(alpha, beta):
    fit1 = holt.fit(smoothing_level=alpha, smoothing_trend=beta)
    fcast1 = fit1.forecast(len(y_test))
    y_test.plot(color="pink", label='Test')
    fcast1.plot(color="purple", label='Forecast')
    rmse = root_mean_squared_error(y_test, fcast1)
    plt.title(f"RMSE = {rmse:.2f}, alpha = {alpha:.2f}, beta = {beta:.2f}")
    plt.legend(loc='best')
    plt.show()
widgets.interact(holt_linear, alpha=(0.01, 1, 0.01), beta=(0.01, 1, 0.01))

# Holt's Exponential Trend
holt = Holt(y_train, exponential=True)
def holt_exp(alpha, beta):
    fit1 = holt.fit(smoothing_level=alpha, smoothing_trend=beta)
    fcast1 = fit1.forecast(len(y_test))
    y_test.plot(color="pink", label='Test')
    fcast1.plot(color="purple", label='Forecast')
    rmse = root_mean_squared_error(y_test, fcast1)
    plt.title(f"RMSE = {rmse:.2f}, alpha = {alpha:.2f}, beta = {beta:.2f}")
    plt.legend(loc='best')
    plt.show()
widgets.interact(holt_exp, alpha=(0.01, 1, 0.01), beta=(0.01, 1, 0.01))

def holt_both(alpha, beta, exponentiality):
    holt = Holt(y_train, exponential=exponentiality)
    fit1 = holt.fit(smoothing_level=alpha, smoothing_trend=beta)
    fcast1 = fit1.forecast(len(y_test))
    y_test.plot(color="pink", label='Test')
    fcast1.plot(color="purple", label='Forecast')
    rmse = root_mean_squared_error(y_test, fcast1)
    plt.title(f"RMSE = {rmse:.2f}, alpha = {alpha:.2f}, beta = {beta:.2f}")
    plt.legend(loc='best')
    plt.show()
widgets.interact(holt_both, alpha=(0.01, 1, 0.01), beta=(0.01, 1, 0.01),
                 exponentiality=[True, False])

# Damped Methods
def damped(alpha, beta, phi, exponentiality, dampness):
    holt = Holt(y_train, exponential=exponentiality, damped_trend=dampness)
    fit1 = holt.fit(smoothing_level=alpha, smoothing_trend=beta, damping_trend=phi)
    fcast1 = fit1.forecast(len(y_test))
    y_test.plot(color="pink", label='Test')
    fcast1.plot(color="purple", label='Forecast')
    rmse = root_mean_squared_error(y_test, fcast1)
    plt.title(f"RMSE = {rmse:.2f}, alpha = {alpha:.2f}, beta = {beta:.2f}")
    plt.legend(loc='best')
    plt.show()
widgets.interact(damped, alpha=(0.01, 1, 0.01), beta=(0.01, 1, 0.01),phi=(0.01, 1, 0.01),
                 exponentiality=[True, False], dampness=[True, False])

# Holt-Winters Methods

from statsmodels.tsa.api import ExponentialSmoothing

def hw(alpha, beta, gamma, seasonality, periods=12):
    holt = ExponentialSmoothing(y_train, trend='add', seasonal=seasonality,seasonal_periods=periods)
    fit1 = holt.fit(smoothing_level=alpha, smoothing_trend=beta, smoothing_seasonal=gamma)
    fcast1 = fit1.forecast(len(y_test))
    y_test.plot(color="pink", label='Test')
    fcast1.plot(color="purple", label='Forecast')
    rmse = root_mean_squared_error(y_test, fcast1)
    plt.title(f"RMSE={rmse:.2f}, alpha={alpha:.2f}, beta={beta:.2f}, gamma={gamma:.2f}")
    plt.legend(loc='best')
    plt.show()
widgets.interact(hw, alpha=(0.01, 1, 0.01), beta=(0.01, 1, 0.01),gamma=(0.01, 1, 0.01),
                 seasonality=['add', 'mul'])

In [ ]:
# Augmented Dicky Fuller Test

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

result = adfuller(df['Value'], maxlag=10)
print("P-Value =", result[1])
if result[1] < 0.05:
    print("Time Series is Stationary")
else:
    print("Time Series is not Stationary")


#to make data stationary 
ord_1_diff = df['Value'].diff() # does first order differencing
ord_1_diff = ord_1_diff.dropna()
result = adfuller(ord_1_diff, maxlag=10)
print("P-Value =", result[1])
if result[1] < 0.05:
    print("Time Series is Stationary")
else:
    print("Time Series is not Stationary")


plot_acf(df['Value'], lags=30,alpha=None)
plt.show()

beer = pd.read_csv("monthly-beer-production-in-austr.csv")
plot_acf(beer['Monthly beer production'], lags=30,alpha=None)
plt.show()

ARIMA

In [ ]:
model = ARIMA(y_train,order=(1,1,1))
model_fit = model.fit()
#print('Coefficients: %s' % model_fit.params)
predictions = model_fit.predict(start=len(y_train), end=len(y_train)+len(y_test)-1)

y_test.plot(color="pink", label='Test')
predictions.plot(color="purple", label='Forecast')
rmse = root_mean_squared_error(y_test, predictions)
plt.title(f"RMSE={rmse:.5f}")
plt.legend(loc='best')
plt.show()

def arima(p,d,q):
    model = ARIMA(y_train,order=(p,d,q))
    model_fit = model.fit()
    predictions = model_fit.predict(start=len(y_train), end=len(y_train)+len(y_test)-1)
    y_test.plot(color="pink", label='Test')
    predictions.plot(color="purple", label='Forecast')
    rmse = root_mean_squared_error(y_test, predictions)
    plt.title(f"RMSE={rmse:.5f}")
    plt.legend(loc='best')
    plt.show()
widgets.interact( arima, p=(0,5,1), d=(0,5,1), q=(0,5,1) )


#AUTO ARIMA
from pmdarima.arima import auto_arima

model = auto_arima(y_train, trace=True, error_action='ignore', suppress_warnings=True)

predictions = model.predict(n_periods=len(y_test))
predictions = pd.Series(predictions,index = y_test.index)
rmse = root_mean_squared_error(y_test, predictions)
y_test.plot(color="pink", label='Test')
predictions.plot(color="purple", label='Forecast')
plt.title(f"RMSE={rmse:.5f}")
plt.legend(loc='best')
plt.show()

## Seasonal ARIMA
model = ARIMA(y_train,order=(1,1,1), seasonal_order=(1,1,1,12))
model_fit = model.fit()
predictions = model_fit.predict(start=len(y_train), end=len(y_train)+len(y_test)-1)
y_test.plot(color="pink", label='Test')
predictions.plot(color="purple", label='Forecast')
rmse = root_mean_squared_error(y_test, predictions)
plt.title(f"RMSE={rmse:.5f}")
plt.legend(loc='best')
plt.show()

def sarima(p,d,q, P, D, Q, S):
    model = ARIMA(y_train,order=(p,d,q),seasonal_order=(P, D, Q, S))
    model_fit = model.fit()
    predictions = model_fit.predict(start=len(y_train), end=len(y_train)+len(y_test)-1)
    y_test.plot(color="pink", label='Test')
    predictions.plot(color="purple", label='Forecast')
    rmse = root_mean_squared_error(y_test, predictions)
    plt.title(f"RMSE={rmse:.5f}")
    plt.legend(loc='best')
    plt.show()
widgets.interact( sarima, p=(0,5,1), d=(0,5,1), q=(0,5,1),
                 P=(0,5,1), D=(0,5,1), Q=(0,5,1) , S=12 )